# Problems & Goal

- Bài toán cốt lõi: Phân loại đơn nhãn (single-label) 6 lớp theo đúng benchmark. Mặc dù thực tế các lớp có sự chồng chéo ngữ nghĩa (ví dụ: ảnh Tết thường bao gồm cảnh tụ họp), ta sẽ xử lý sự mơ hồ này trực tiếp ở khâu huấn luyện và phân tích lỗi thay vì thay đổi định nghĩa bài toán.
- Vai trò của lớp "other": Hoạt động như một phễu lọc (catch-all failure mode) để hứng các mẫu ngoại lai hoặc không rõ ràng, chứ không mang một đặc trưng ngữ nghĩa độc lập.
- Thách thức từ dữ liệu: Kích thước tập mẫu cực nhỏ, mất cân bằng nghiêm trọng, nhãn nhiễu và ranh giới phân loại mờ nhạt (ví dụ: ảnh công viên dễ nhầm thành thiên nhiên).
- Tiêu chí tối ưu: Ưu tiên tính mạnh mẽ (robustness) và độ tin cậy khi đưa vào thực tế. Không chạy đua tối ưu độ chính xác điểm (point accuracy) trên tập dữ liệu nhỏ vốn rất dễ bị overfit.
- Chiến lược thực thi: Tiếp cận theo hướng tối ưu dữ liệu (data-centric) thay vì dùng mạng end-to-end phức tạp. Giải pháp là sử dụng Frozen Embeddings (SigLIP2 đa ngôn ngữ) kết hợp Classification Head có trọng số, đi kèm kỹ thuật hiệu chuẩn (calibration) và lọc nhiễu offline (CLIPCleaner).

# EDA

## Manifest

In [ ]:
from pathlib import Path

from z_photos.conf import Config

cfg = Config(
    data_root=Path("../data"),
    model_name_or_path="google/siglip2-so400m-patch16-384",
    seed=42,
)

## Bronze -> Silver: FiftyOne Dataset

TODO(Explain why FiftyOne)

In [ ]:
import z_photos
from z_photos.datasets import build_dataset

dataset = build_dataset(z_photos.__name__, bronze_dir=cfg.bronze_dir)
print(dataset)

## FiftyOne App

> **Tại sao dùng FiftyOne cho EDA?**
> - **Visual browsing + tagging** nhanh hơn bất kỳ tool nào khác: click → xem ảnh, gắn tag ngay trên UI.
> - **Embedding Panel** liên kết trực tiếp UMAP plot ↔ ảnh gốc ↔ label.
> - **Brain API** cung cấp exact duplicates, near-duplicate, leaky splits, uniqueness, và mistakenness — tất cả không cần viết lại từ đầu.
>
> **Convention:** dataset gốc tên `z_photos` (persistent) không bị sửa. Mọi thao tác EDA/brain dùng clone tên `z_photos_eda` (non-persistent, tự xóa khi kernel restart).

In [ ]:
import fiftyone as fo

EDA_DATASET_NAME = "z_photos_eda"

# Clone dataset chỉ để EDA, không persistent (tự xóa khi kernel restart).
# Không sửa dataset gốc "z_photos".
if EDA_DATASET_NAME in fo.list_datasets():
    fo.delete_dataset(EDA_DATASET_NAME)

eda = dataset.clone(name=EDA_DATASET_NAME, persistent=False)
print(eda)

## Dataset Inventory & Sanity Checks

Kiểm tra toàn diện trước khi train:
1. **Class distribution** — mất cân bằng nghiêm trọng giữa train/test?
2. **Metadata sanity** — kích thước ảnh, file corrupt?
3. **Exact duplicates** — file trùng lặp chính xác (theo hash).
4. **Near duplicates** — ảnh gần giống nhau về nội dung.
5. **Leaky splits** — ảnh train/test quá giống nhau → inflate metric.

In [ ]:
from collections import Counter

import pandas as pd

# ── 1. Class distribution per split
train_view = eda.match_tags("train")
test_view = eda.match_tags("test")

train_counts = Counter(train_view.values("ground_truth.label"))
test_counts = Counter(test_view.values("ground_truth.label"))

all_classes = sorted(set(train_counts) | set(test_counts))
df_dist = pd.DataFrame(
    {
        "class": all_classes,
        "train": [train_counts.get(c, 0) for c in all_classes],
        "test": [test_counts.get(c, 0) for c in all_classes],
    }
).set_index("class")
df_dist["total"] = df_dist["train"] + df_dist["test"]
df_dist["train_pct"] = (df_dist["train"] / df_dist["train"].sum() * 100).round(1)
df_dist["test_pct"] = (df_dist["test"] / df_dist["test"].sum() * 100).round(1)

print("=== Class Distribution ===")
print(df_dist.to_string())
print(f"\nTrain total: {df_dist['train'].sum()} | Test total: {df_dist['test'].sum()}")

In [ ]:
import numpy as np

# ── 2. Metadata sanity: resolution, aspect ratio, corrupt check
widths = eda.values("metadata.width")
heights = eda.values("metadata.height")
sizes = eda.values("metadata.size_bytes")

# Detect sample với metadata bị None (corrupt hoặc unreadable)
missing_meta = sum(1 for w in widths if w is None)
print(f"Samples với metadata thiếu (có thể corrupt): {missing_meta}")

w_arr = np.array([w for w in widths if w], dtype=float)
h_arr = np.array([h for h in heights if h], dtype=float)
s_arr = np.array([s for s in sizes if s], dtype=float)
ar_arr = w_arr / h_arr

print("\n=== Resolution Stats ===")
print(f"Width:  min={w_arr.min():.0f}  max={w_arr.max():.0f}  mean={w_arr.mean():.0f}")
print(f"Height: min={h_arr.min():.0f}  max={h_arr.max():.0f}  mean={h_arr.mean():.0f}")
print(
    f"Aspect ratio: min={ar_arr.min():.2f}  max={ar_arr.max():.2f}"
    f"  mean={ar_arr.mean():.2f}"
)
print(
    f"File size (KB): min={s_arr.min() / 1024:.1f}  max={s_arr.max() / 1024:.1f}"
    f"  mean={s_arr.mean() / 1024:.1f}"
)

### Exact & Near Duplicates

`fob.compute_exact_duplicates` — so sánh hash file, phát hiện file trùng chính xác.  
`fob.compute_near_duplicates` — dùng embedding similarity, phát hiện ảnh gần giống nhau về nội dung.

In [ ]:
import fiftyone.brain as fob

# ── Exact duplicates (hash-based, không cần model) ──────
# Returns dict: {representative_id: [dup_id, ...]}
exact_dups = fob.compute_exact_duplicates(eda, progress=False)

total_dup_samples = sum(len(v) for v in exact_dups.values())
print(f"Exact duplicate groups: {len(exact_dups)}")
print(f"Exact duplicate samples (to remove): {total_dup_samples}")

if exact_dups:
    for rep_id, dup_ids in list(exact_dups.items())[:3]:
        rep = eda[rep_id]
        fname = rep.filepath.split("/")[-1]
        print(f"\n  Representative [{rep.tags}] {rep.ground_truth.label}: .../{fname}")
        for did in dup_ids:
            s = eda[did]
            fname = s.filepath.split("/")[-1]
            print(f"    Duplicate [{s.tags}] {s.ground_truth.label}: .../{fname}")

In [ ]:
# ── Near duplicates (embedding-based, dùng mobilenet nhẹ)
# Bước 1: Tính similarity index (có brain_key để tái dùng)
sim_index = fob.compute_similarity(
    eda,
    brain_key="near_dups_sim",
    model="mobilenet-v2-imagenet-torch",
    backend="sklearn",
    metric="cosine",
    progress=False,
)

# Bước 2: Tìm near duplicates từ similarity index đã tính
near_dup_index = fob.compute_near_duplicates(
    eda,
    similarity_index=sim_index,
    threshold=0.1,  # cosine distance < 0.1 → very similar
    progress=False,
)

dup_ids = near_dup_index.duplicate_ids
print(f"Near duplicate samples (threshold=0.1): {len(dup_ids)}")
rate = len(dup_ids) / len(eda) * 100
print(f"Near-dup rate: {len(dup_ids)}/{len(eda)} = {rate:.1f}%")

if dup_ids:
    dup_samples = eda.select(dup_ids)
    dup_label_counts = Counter(dup_samples.values("ground_truth.label"))
    dup_tag_counts = Counter([t for ts in dup_samples.values("tags") for t in ts])
    print(f"\nNear-dups by class: {dict(sorted(dup_label_counts.items()))}")
    print(f"Near-dups by split: {dict(dup_tag_counts)}")

### Leaky Splits Detection

Tìm ảnh train và test quá giống nhau về nội dung → **data leakage** → metric bị inflate.  
Với dataset nhỏ (~261 train / 60 test), vài ảnh trùng có thể làm accuracy tăng giả tạo.

In [ ]:
# ── Leaky Splits: tìm ảnh train/test quá giống nhau
# Tái dùng similarity index đã tính ở bước trên (brain_key="near_dups_sim")
leaky_index = fob.compute_leaky_splits(
    eda,
    splits=["train", "test"],
    similarity_index=sim_index,
    progress=False,
)

leaks_view = leaky_index.leaks_view()
print(f"Leaky samples (train↔test near-duplicates): {len(leaks_view)}")

if len(leaks_view) > 0:
    print("\nCác cặp bị leak (train sample → test sample giống nhau):")
    for sample in leaks_view.iter_samples():
        fname = sample.filepath.split("/")[-1]
        print(f"  [{sample.tags}] {sample.ground_truth.label}: .../{fname}")
else:
    print("✓ Không phát hiện leaky splits — train/test tách biệt tốt.")

### EDA: Uniqueness & FiftyOne App

`fob.compute_uniqueness` — xếp hạng độ "độc bản" của từng ảnh (thấp = redundant, cao = outlier).  
`fo.launch_app` — mở giao diện tương tác để visual browsing, tagging, Embeddings Panel.

In [ ]:
# ── Uniqueness: tìm ảnh redundant và outlier
# Tái dùng mobilenet embeddings từ similarity index
fob.compute_uniqueness(
    eda,
    similarity_index=sim_index,
    uniqueness_field="uniqueness",
    progress=False,
)

uniqueness_vals = eda.values("uniqueness")
uniqueness_arr = np.array([v for v in uniqueness_vals if v is not None])

print("Uniqueness stats:")
print(
    f"  min={uniqueness_arr.min():.3f}  max={uniqueness_arr.max():.3f}"
    f"  mean={uniqueness_arr.mean():.3f}"
)

# Top-5 most unique (potential outliers)
print("\nTop-5 most unique samples (potential outliers/OOD):")
top5_view = eda.sort_by("uniqueness", reverse=True).limit(5)
for s in top5_view.iter_samples():
    fname = s.filepath.split("/")[-1]
    print(f"  u={s.uniqueness:.3f} [{s.tags}] {s.ground_truth.label}: .../{fname}")

# Top-5 least unique (most redundant)
print("\nTop-5 least unique samples (most redundant):")
bot5_view = eda.sort_by("uniqueness", reverse=False).limit(5)
for s in bot5_view.iter_samples():
    fname = s.filepath.split("/")[-1]
    print(f"  u={s.uniqueness:.3f} [{s.tags}] {s.ground_truth.label}: .../{fname}")

# ── FiftyOne App: visual browsing
# Mở App để:
#   • Sidebar: xem phân phối class, filter theo split/class
#   • Sort by "uniqueness" → review outlier/OOD
#   • Embeddings panel: chọn "near_dups_sim" → UMAP, Color by "ground_truth.label"
#   • Tag ảnh nghi ngờ: click ảnh → nhấn "t" → nhập tag "noisy" hoặc "ambiguous"

In [ ]:
session = fo.launch_app(eda, auto=False)
print("FiftyOne App: http://localhost:5151")

## Baseline: Zero-shot